# 04. RLHF 이론

## 학습 목표
- RLHF 파이프라인의 3단계 (SFT → Reward Model → PPO) 이해
- Reward Model의 수학적 원리 (Bradley-Terry 모델) 이해
- PPO의 핵심 아이디어와 KL-divergence 제약 이해
- RLHF의 한계와 대안 (DPO) 파악

## 핵심 논문
- [InstructGPT (Ouyang et al., 2022)](https://arxiv.org/abs/2203.02155)
- [Learning to summarize from human feedback (Stiennon et al., 2020)](https://arxiv.org/abs/2009.01325)

---

In [ ]:
# Google Colab 환경 설정
!pip install -q transformers datasets accelerate torch matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. RLHF 파이프라인 전체 그림

RLHF (Reinforcement Learning from Human Feedback)는 LLM을 **인간의 선호에 맞추는** 방법이다.

### 왜 RLHF가 필요한가?

SFT (Supervised Fine-Tuning)만으로는 부족한 이유:
- SFT는 **하나의 정답**에 맞추는 학습 → 다양한 좋은 답변을 구분하지 못함
- 인간의 선호는 "맞다/틀리다"보다 "A가 B보다 낫다"로 표현됨
- **Helpfulness, Harmlessness, Honesty** (HHH) 같은 추상적 기준을 loss function으로 만들기 어려움

### 3단계 파이프라인

```
Step 1: SFT                Step 2: Reward Model          Step 3: RL (PPO)
━━━━━━━━━━━━━━━         ━━━━━━━━━━━━━━━━━━━━         ━━━━━━━━━━━━━━━━
고품질 데이터로            인간 선호 데이터로              Reward Model을 보상으로
지시 따르기 학습           보상 모델 학습                 정책(LLM) 최적화

Input: (prompt, response)  Input: (prompt, win, lose)    Input: prompt
Output: SFT 모델           Output: Reward Model          Output: Aligned 모델
```

In [ ]:
# RLHF 파이프라인 시각화
fig, ax = plt.subplots(figsize=(14, 6))

# 배경 박스들
boxes = [
    {'xy': (0.5, 2), 'text': 'Step 1: SFT\n\nBase Model\n+ Instruction Data\n→ SFT Model', 'color': '#E3F2FD'},
    {'xy': (5, 2), 'text': 'Step 2: Reward Model\n\nSFT Model outputs\n+ Human Preferences\n→ Reward Model', 'color': '#FFF3E0'},
    {'xy': (9.5, 2), 'text': 'Step 3: PPO\n\nSFT Model (policy)\n+ Reward Model\n→ Aligned Model', 'color': '#E8F5E9'},
]

for box in boxes:
    rect = mpatches.FancyBboxPatch(box['xy'], 3.5, 3, boxstyle='round,pad=0.2',
                                    facecolor=box['color'], edgecolor='gray', linewidth=2)
    ax.add_patch(rect)
    ax.text(box['xy'][0] + 1.75, box['xy'][1] + 1.5, box['text'],
            ha='center', va='center', fontsize=10, fontweight='bold')

# 화살표
ax.annotate('', xy=(5, 3.5), xytext=(4, 3.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(9.5, 3.5), xytext=(8.5, 3.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# 데이터 표시
ax.text(2.25, 0.8, 'Instruction\nDataset', ha='center', va='center',
        fontsize=9, style='italic', color='blue',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='blue', alpha=0.8))
ax.text(6.75, 0.8, 'Human Preference\n(win vs lose)', ha='center', va='center',
        fontsize=9, style='italic', color='orange',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='orange', alpha=0.8))
ax.text(11.25, 0.8, 'Reward Signal\n+ KL Constraint', ha='center', va='center',
        fontsize=9, style='italic', color='green',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='green', alpha=0.8))

ax.annotate('', xy=(2.25, 2), xytext=(2.25, 1.3),
            arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
ax.annotate('', xy=(6.75, 2), xytext=(6.75, 1.3),
            arrowprops=dict(arrowstyle='->', color='orange', lw=1.5))
ax.annotate('', xy=(11.25, 2), xytext=(11.25, 1.3),
            arrowprops=dict(arrowstyle='->', color='green', lw=1.5))

ax.set_xlim(-0.5, 14)
ax.set_ylim(0, 6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('RLHF Pipeline: SFT → Reward Model → PPO', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

---
## 2. Step 1 - SFT: Supervised Fine-Tuning

이미 02번 노트북에서 다룬 내용. RLHF의 출발점.

- 고품질 instruction-response 데이터로 모델을 Fine-tuning
- InstructGPT에서는 약 13,000개의 고품질 데이터 사용
- SFT 모델이 RLHF의 **초기 정책(policy)**이 됨

$$\mathcal{L}_{\text{SFT}} = -\sum_{t} \log P_{\theta}(y_t | x, y_{<t})$$

---
## 3. Step 2 - Reward Model: 인간 선호 데이터로 보상 학습

### 인간 선호 데이터

같은 프롬프트에 대해 두 응답 중 **어떤 것이 더 좋은지** 인간이 판별:

```
Prompt: "Explain quantum computing simply."

Response A (chosen):  "Quantum computing uses quantum bits that can be 
                       0 and 1 simultaneously, enabling parallel computation."

Response B (rejected): "Quantum computing is a type of computation that 
                        uses quantum-mechanical phenomena."

Human label: A > B (A가 더 좋다)
```

### Bradley-Terry 모델

두 응답의 선호 확률을 모델링:

$$P(y_w \succ y_l | x) = \sigma(r_\theta(x, y_w) - r_\theta(x, y_l))$$

여기서:
- $y_w$: 선호된 응답 (winner/chosen)
- $y_l$: 비선호 응답 (loser/rejected)
- $r_\theta(x, y)$: Reward Model이 예측한 보상 스칼라
- $\sigma$: sigmoid 함수

### Reward Model 학습 목표

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma(r_\theta(x, y_w) - r_\theta(x, y_l)) \right]$$

직관: **선호된 응답의 보상이 비선호 응답보다 높도록** 학습

In [ ]:
# Bradley-Terry 모델 시각화

# sigma(r_w - r_l) 함수 시각화
diff = np.linspace(-5, 5, 100)
prob = 1 / (1 + np.exp(-diff))  # sigmoid

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 왼쪽: sigmoid(r_w - r_l)
ax = axes[0]
ax.plot(diff, prob, 'b-', linewidth=2)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(diff, prob, 0.5, where=(diff > 0), alpha=0.1, color='green', label='P(win) > 0.5')
ax.fill_between(diff, prob, 0.5, where=(diff < 0), alpha=0.1, color='red', label='P(win) < 0.5')
ax.set_xlabel('r(chosen) - r(rejected)', fontsize=11)
ax.set_ylabel('P(chosen > rejected)', fontsize=11)
ax.set_title('Bradley-Terry Model', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# 오른쪽: Loss = -log(sigma(r_w - r_l))
ax = axes[1]
loss = -np.log(prob + 1e-8)
ax.plot(diff, loss, 'r-', linewidth=2)
ax.set_xlabel('r(chosen) - r(rejected)', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Reward Model Loss', fontsize=13)
ax.annotate('Loss가 높음\n(잘못 예측)',
            xy=(-3, -np.log(1/(1+np.exp(3)))),
            xytext=(-4, 4), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='red'))
ax.annotate('Loss가 낮음\n(올바른 예측)',
            xy=(3, -np.log(1/(1+np.exp(-3)))),
            xytext=(1, 2), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='green'))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Reward Model 구현: 선호 데이터로 학습하는 과정

간단한 Reward Model을 직접 구현해보자.

In [ ]:
# 간단한 Reward Model 구현

class SimpleRewardModel(nn.Module):
    """텍스트 임베딩을 받아 스칼라 보상을 출력하는 모델"""
    
    def __init__(self, input_dim=768):
        super().__init__()
        self.reward_head = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)  # 스칼라 보상 출력
        )
    
    def forward(self, embeddings):
        return self.reward_head(embeddings).squeeze(-1)


def reward_model_loss(reward_chosen, reward_rejected):
    """
    Bradley-Terry loss
    L = -log(sigma(r_chosen - r_rejected))
    """
    return -F.logsigmoid(reward_chosen - reward_rejected).mean()


print("Reward Model Loss 함수 동작 확인:")
r_chosen = torch.tensor([2.0, 1.5, 3.0])
r_rejected = torch.tensor([1.0, 0.5, 0.0])

loss = reward_model_loss(r_chosen, r_rejected)
print(f"  chosen rewards: {r_chosen.tolist()}")
print(f"  rejected rewards: {r_rejected.tolist()}")
print(f"  차이 (chosen - rejected): {(r_chosen - r_rejected).tolist()}")
print(f"  Loss: {loss.item():.4f} (chosen > rejected이므로 낮은 loss)")

# 반대 경우
loss_wrong = reward_model_loss(r_rejected, r_chosen)
print(f"\n  반대로 하면 Loss: {loss_wrong.item():.4f} (높은 loss!)")

In [ ]:
# 간단한 선호 데이터로 Reward Model 학습

# 시뮬레이션: 임베딩 공간에서 선호 데이터 생성
# 실제로는 BERT/LLM의 출력 임베딩을 사용
torch.manual_seed(42)

n_samples = 200
embed_dim = 64

# 좋은 응답: 특정 방향의 임베딩이 높은 값
# 나쁜 응답: 랜덤 임베딩
chosen_embeddings = torch.randn(n_samples, embed_dim) + 0.5   # 약간 양의 방향
rejected_embeddings = torch.randn(n_samples, embed_dim) - 0.5  # 약간 음의 방향

# Reward Model 학습
reward_model = SimpleRewardModel(input_dim=embed_dim)
optimizer = optim.Adam(reward_model.parameters(), lr=1e-3)

losses = []
accuracies = []

for epoch in range(100):
    reward_model.train()
    
    r_chosen = reward_model(chosen_embeddings)
    r_rejected = reward_model(rejected_embeddings)
    
    loss = reward_model_loss(r_chosen, r_rejected)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # 정확도: chosen reward > rejected reward인 비율
    acc = (r_chosen > r_rejected).float().mean().item()
    
    losses.append(loss.item())
    accuracies.append(acc)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:>3d}: Loss = {loss.item():.4f}, Accuracy = {acc:.4f}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses, 'b-', alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Reward Model Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(accuracies, 'g-', alpha=0.7)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Reward Model Accuracy (chosen > rejected)')
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 학습된 Reward Model의 보상 분포 확인
reward_model.eval()
with torch.no_grad():
    r_c = reward_model(chosen_embeddings).numpy()
    r_r = reward_model(rejected_embeddings).numpy()

plt.figure(figsize=(8, 4))
plt.hist(r_c, bins=30, alpha=0.6, label='Chosen (preferred)', color='green')
plt.hist(r_r, bins=30, alpha=0.6, label='Rejected', color='red')
plt.xlabel('Reward Score')
plt.ylabel('Count')
plt.title('Reward Distribution: Chosen vs Rejected')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Chosen 평균 보상: {r_c.mean():.4f}")
print(f"Rejected 평균 보상: {r_r.mean():.4f}")
print(f"차이: {r_c.mean() - r_r.mean():.4f}")
print(f"정확도: {(r_c > r_r.mean()).mean():.4f}")

---
## 5. Step 3 - PPO: Proximal Policy Optimization

### RL 관점에서의 LLM

| RL 개념 | LLM에서의 의미 |
|---------|---------------|
| Policy $\pi_\theta$ | LLM (텍스트 생성 모델) |
| State $s$ | 현재까지 생성된 토큰 시퀀스 |
| Action $a$ | 다음에 생성할 토큰 |
| Reward $r$ | Reward Model이 전체 응답에 부여하는 점수 |
| Environment | 프롬프트 + 생성 과정 |

### PPO 목적 함수

$$\mathcal{L}_{\text{PPO}} = \mathbb{E}_{x \sim D, y \sim \pi_\theta} \left[ r_\phi(x, y) - \beta \cdot D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}}) \right]$$

직관:
- $r_\phi(x, y)$: Reward Model의 보상을 **최대화** (좋은 응답 생성)
- $\beta \cdot D_{\text{KL}}$: SFT 모델에서 너무 멀어지지 않도록 **제약**

### PPO Clipped Objective

$$\mathcal{L}_{\text{CLIP}} = \mathbb{E}_t \left[ \min\left( \frac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)} \hat{A}_t, \; \text{clip}\left(\frac{\pi_\theta}{\pi_{\text{old}}}, 1-\epsilon, 1+\epsilon\right) \hat{A}_t \right) \right]$$

- $\hat{A}_t$: Advantage (보상 - 기대 보상)
- $\epsilon$: 클리핑 범위 (보통 0.2)
- 정책이 한 번에 너무 크게 변하지 않도록 제한

In [ ]:
# PPO Clipping 시각화

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
epsilon = 0.2

# 왼쪽: Advantage > 0 (좋은 행동을 더 많이 하고 싶을 때)
ax = axes[0]
ratio = np.linspace(0.3, 2.0, 200)
A = 1.0  # Positive advantage

objective_unclipped = ratio * A
objective_clipped = np.clip(ratio, 1 - epsilon, 1 + epsilon) * A
objective_ppo = np.minimum(objective_unclipped, objective_clipped)

ax.plot(ratio, objective_unclipped, '--', color='gray', alpha=0.5, label='Unclipped')
ax.plot(ratio, objective_clipped, '--', color='orange', alpha=0.5, label='Clipped')
ax.plot(ratio, objective_ppo, 'b-', linewidth=2, label='PPO (min)')
ax.axvline(x=1, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=1-epsilon, color='red', linestyle=':', alpha=0.5, label=f'1-eps={1-epsilon}')
ax.axvline(x=1+epsilon, color='red', linestyle=':', alpha=0.5, label=f'1+eps={1+epsilon}')
ax.set_xlabel('Policy Ratio (pi_new / pi_old)')
ax.set_ylabel('Objective')
ax.set_title(f'PPO Clipping: Advantage > 0 (A={A})')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 오른쪽: Advantage < 0 (나쁜 행동을 줄이고 싶을 때)
ax = axes[1]
A = -1.0  # Negative advantage

objective_unclipped = ratio * A
objective_clipped = np.clip(ratio, 1 - epsilon, 1 + epsilon) * A
objective_ppo = np.minimum(objective_unclipped, objective_clipped)

ax.plot(ratio, objective_unclipped, '--', color='gray', alpha=0.5, label='Unclipped')
ax.plot(ratio, objective_clipped, '--', color='orange', alpha=0.5, label='Clipped')
ax.plot(ratio, objective_ppo, 'b-', linewidth=2, label='PPO (min)')
ax.axvline(x=1, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=1-epsilon, color='red', linestyle=':', alpha=0.5)
ax.axvline(x=1+epsilon, color='red', linestyle=':', alpha=0.5)
ax.set_xlabel('Policy Ratio (pi_new / pi_old)')
ax.set_ylabel('Objective')
ax.set_title(f'PPO Clipping: Advantage < 0 (A={A})')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("PPO Clipping의 효과:")
print("  - Advantage > 0: ratio가 1+eps를 넘으면 더 이상 보상이 증가하지 않음")
print("  - Advantage < 0: ratio가 1-eps 아래로 내려가면 더 이상 페널티가 증가하지 않음")
print("  → 정책이 한 번에 너무 크게 변하는 것을 방지!")

---
## 6. KL-divergence 제약: 왜 base 모델에서 너무 멀어지면 안 되는가

### KL-divergence

$$D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}}) = \mathbb{E}_{y \sim \pi_\theta} \left[ \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} \right]$$

### 왜 필요한가?

KL 제약이 없으면:
1. **Reward Hacking**: Reward Model의 취약점을 악용하여 높은 보상을 받지만 실제로는 나쁜 응답 생성
2. **Mode Collapse**: 소수의 "안전한" 응답만 반복 생성
3. **언어 능력 저하**: 문법, 유창성 등 기본 능력을 잃음

### Beta ($\beta$) 하이퍼파라미터

- $\beta$가 크면: 보수적 (SFT 모델에 가까움, 안정적이지만 개선 제한)
- $\beta$가 작으면: 공격적 (많이 변할 수 있음, Reward Hacking 위험)
- 보통 $\beta = 0.01 \sim 0.2$ 범위에서 설정

In [ ]:
# KL-divergence 제약 효과 시각화

# 시뮬레이션: Reward vs KL tradeoff
np.random.seed(42)

betas = [0.001, 0.01, 0.05, 0.1, 0.5]
training_steps = 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for beta in betas:
    rewards = []
    kls = []
    reward = 0
    kl = 0
    
    for step in range(training_steps):
        # 시뮬레이션: reward는 증가하려 하고, KL도 증가
        reward_gain = 0.05 * (1 - reward / 5)  # 보상 증가 (포화)
        kl_increase = 0.02 * (1 + reward)  # KL 증가 (보상과 함께)
        
        # beta가 크면 KL 페널티가 커서 업데이트가 작아짐
        effective_gain = reward_gain - beta * kl_increase
        reward += max(0, effective_gain * 0.5)
        kl += kl_increase * (1 - beta)
        
        rewards.append(reward)
        kls.append(kl)
    
    axes[0].plot(range(training_steps), rewards, label=f'beta={beta}')
    axes[1].plot(range(training_steps), kls, label=f'beta={beta}')

axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Reward')
axes[0].set_title('Reward over Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Training Step')
axes[1].set_ylabel('KL Divergence')
axes[1].set_title('KL Divergence from Reference')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Beta의 역할:")
print("  beta 작음 → Reward 높지만 KL도 커짐 (Reward Hacking 위험)")
print("  beta 큼   → KL은 작지만 Reward 개선이 제한적")
print("  → 적절한 beta를 찾는 것이 중요!")

---
## 7. RLHF의 한계와 대안

### RLHF의 한계

| 한계 | 설명 |
|------|------|
| **복잡한 파이프라인** | 3개의 모델 (SFT, RM, Policy) + PPO 학습 |
| **불안정한 학습** | PPO는 하이퍼파라미터에 민감, reward hacking 위험 |
| **높은 비용** | GPU 메모리에 최소 2-3개 모델 동시 로드 |
| **Reward Model 품질** | RM이 잘못되면 전체가 무너짐 |
| **인간 레이블 품질** | 레이블러 간 일관성 문제 |

### RLHF의 메모리 요구량 (7B 모델 기준)

```
Policy Model (학습 중):    ~14 GB
Reference Model (동결):    ~14 GB
Reward Model:              ~14 GB
Value Model:               ~14 GB
Optimizer States:           ~28 GB
─────────────────────────────────
합계:                      ~84+ GB
```

### 대안: DPO (Direct Preference Optimization)

- Reward Model과 PPO 없이 **직접** 선호 데이터로 학습
- 수학적으로 RLHF와 동일한 최적해를 가짐
- 구현이 훨씬 간단하고 안정적

→ 다음 노트북에서 자세히 다룸!

In [ ]:
# RLHF vs DPO 메모리 비교
methods = ['RLHF\n(PPO)', 'DPO']

# 7B 모델 기준 메모리 (GB)
rlhf_memory = {'Policy': 14, 'Reference': 14, 'Reward': 14, 'Value': 14, 'Optimizer': 28}
dpo_memory = {'Policy': 14, 'Reference': 14, 'Optimizer': 28}

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
colors_list = ['#2196F3', '#FF9800', '#F44336', '#4CAF50', '#9C27B0']

# RLHF
ax = axes[0]
values = list(rlhf_memory.values())
labels = list(rlhf_memory.keys())
ax.barh(labels, values, color=colors_list[:len(values)])
for i, v in enumerate(values):
    ax.text(v + 0.5, i, f'{v} GB', va='center', fontweight='bold')
ax.set_xlabel('GPU Memory (GB)')
ax.set_title(f'RLHF (Total: {sum(values)} GB)', fontsize=13)
ax.set_xlim(0, 35)
ax.grid(True, alpha=0.3, axis='x')

# DPO
ax = axes[1]
values = list(dpo_memory.values())
labels = list(dpo_memory.keys())
ax.barh(labels, values, color=colors_list[:len(values)])
for i, v in enumerate(values):
    ax.text(v + 0.5, i, f'{v} GB', va='center', fontweight='bold')
ax.set_xlabel('GPU Memory (GB)')
ax.set_title(f'DPO (Total: {sum(values)} GB)', fontsize=13)
ax.set_xlim(0, 35)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"RLHF: {sum(rlhf_memory.values())} GB 필요")
print(f"DPO:  {sum(dpo_memory.values())} GB 필요")
print(f"절약: {sum(rlhf_memory.values()) - sum(dpo_memory.values())} GB ({(1-sum(dpo_memory.values())/sum(rlhf_memory.values()))*100:.0f}%)")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: BERT 기반 Reward Model 학습

BERT를 backbone으로 사용하는 Reward Model을 구현하고, 실제 텍스트 쌍으로 학습하세요.
- 텍스트 입력 → BERT 임베딩 → 스칼라 보상
- Bradley-Terry loss로 학습
- 학습 후 새로운 텍스트 쌍에 대한 선호 예측

In [ ]:
# TODO: BERT 기반 Reward Model 구현
# Hint:
# 1. BERT를 backbone으로 사용 (AutoModel.from_pretrained('bert-base-uncased'))
# 2. [CLS] 토큰 임베딩 → Linear → 스칼라 보상
# 3. 선호 데이터 생성:
#    chosen: 정확하고 유용한 답변
#    rejected: 부정확하거나 모호한 답변
# 4. Bradley-Terry loss로 학습
# 5. 학습 곡선과 정확도 그래프 그리기

# class BERTRewardModel(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.bert = AutoModel.from_pretrained('bert-base-uncased')
#         self.reward_head = nn.Linear(768, 1)
#     
#     def forward(self, input_ids, attention_mask):
#         outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
#         cls_embedding = outputs.last_hidden_state[:, 0, :]
#         return self.reward_head(cls_embedding).squeeze(-1)

# preference_data = [
#     {'prompt': 'What is AI?',
#      'chosen': 'AI is a field of computer science that creates intelligent systems.',
#      'rejected': 'AI is something with computers.'},
#     ...
# ]

---
## 핵심 정리

| 개념 | 설명 | 핵심 포인트 |
|------|------|-------------|
| RLHF 파이프라인 | SFT → Reward Model → PPO | 3단계로 인간 선호에 맞춤 |
| Bradley-Terry 모델 | $P(w>l) = \sigma(r_w - r_l)$ | 쌍별 비교로 선호 모델링 |
| Reward Model | 응답에 스칼라 보상 부여 | 인간 선호를 수치화 |
| PPO | 정책 업데이트를 안정적으로 제한 | Clipping으로 큰 변화 방지 |
| KL Constraint | reference 모델에서 멀어지지 않도록 | Reward Hacking 방지 |
| RLHF 한계 | 복잡, 불안정, 고비용 | DPO가 대안 |

**다음 노트북**: [05-dpo.ipynb](05-dpo.ipynb) - Direct Preference Optimization (RLHF의 간단한 대안)